# Comparativa de Algoritmos Genéticos: Binario vs Real
Este notebook contiene la implementación y comparación de dos enfoques de Algoritmos Genéticos (AG) para optimizar la función Esfera.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Función objetivo: Esfera (Mínimo global en 0,0)
def objective_function(x):
    return np.sum(x**2)

## 1. Algoritmo Genético Binario
Utiliza representación de bits, requiere decodificación, selección por ruleta y cruza de punto simple.

In [ ]:
def binary_to_real(binary_str, bounds, bits_per_var):
    max_bin = 2**bits_per_var - 1
    # Convert binary list to integer
    integer_val = int("".join(map(str, binary_str)), 2)
    return bounds[0] + (integer_val / max_bin) * (bounds[1] - bounds[0])

def genetic_algorithm_binary(pop_size=50, generations=100, bits_per_var=16):
    bounds = [-5.12, 5.12]
    pop = np.random.randint(0, 2, (pop_size, bits_per_var))
    best_history = []

    for _ in range(generations):
        # Decodificación y evaluación
        decoded = np.array([binary_to_real(ind, bounds, bits_per_var) for ind in pop])
        costs = np.array([objective_function(np.array([d])) for d in decoded])
        fitness = 1 / (1 + costs)
        best_history.append(np.min(costs))

        # Selección: Ruleta
        probs = fitness / np.sum(fitness)
        indices = np.random.choice(pop_size, size=pop_size, p=probs)
        pop = pop[indices]

        # Cruza: Punto Simple
        for i in range(0, pop_size - 1, 2):
            if np.random.rand() < 0.8:
                pt = np.random.randint(1, bits_per_var - 1)
                p1, p2 = pop[i].copy(), pop[i+1].copy()
                pop[i] = np.append(p1[:pt], p2[pt:])
                pop[i+1] = np.append(p2[:pt], p1[pt:])

        # Mutación: Flip de Bit
        mask = np.random.rand(*pop.shape) < 0.01
        pop[mask] = 1 - pop[mask]
        
    return best_history

## 2. Algoritmo Genético Real
Opera directamente con flotantes, selección por torneo, cruza aritmética y mutación gaussiana.

In [ ]:
def genetic_algorithm_real(pop_size=50, generations=100):
    bounds = [-5.12, 5.12]
    pop = np.random.uniform(bounds[0], bounds[1], (pop_size, 1))
    best_history = []

    for _ in range(generations):
        costs = np.array([objective_function(ind) for ind in pop])
        best_history.append(np.min(costs))

        # Selección: Torneo
        new_pop = []
        for _ in range(pop_size):
            candidates_idx = np.random.choice(pop_size, 3)
            best_idx = candidates_idx[np.argmin(costs[candidates_idx])]
            new_pop.append(pop[best_idx])
        pop = np.array(new_pop)

        # Cruza: Aritmética
        for i in range(0, pop_size - 1, 2):
            if np.random.rand() < 0.8:
                alpha = np.random.rand()
                p1, p2 = pop[i].copy(), pop[i+1].copy()
                pop[i] = alpha * p1 + (1 - alpha) * p2
                pop[i+1] = alpha * p2 + (1 - alpha) * p1

        # Mutación: Gaussiana
        mutation_mask = np.random.rand(*pop.shape) < 0.05
        noise = np.random.normal(0, 0.1, pop.shape)
        pop += mutation_mask * noise
        pop = np.clip(pop, bounds[0], bounds[1])
        
    return best_history

In [ ]:
gens = 100
hist_bin = genetic_algorithm_binary(generations=gens)
hist_real = genetic_algorithm_real(generations=gens)

plt.figure(figsize=(10, 6))
plt.plot(hist_bin, label='AG Binario (Ruleta/Punto Simple)', color='blue')
plt.plot(hist_real, label='AG Real (Torneo/Aritmética)', color='red')
plt.title('Comparación de Convergencia: Binario vs Real')
plt.xlabel('Generación')
plt.ylabel('Mejor Fitness (Error)')
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", ls="-")
plt.show()